## [消息](https://docs.langchain.com/oss/python/langchain/messages)

#### 分类

1. SystemMessage
2. HumanMessage
3. AIMessage
4. ToolMessage

#### 加载环境并初始化

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model='deepseek:deepseek-flash'
)

#### 聊天机器人

In [ ]:
from typing import List
from langchain_core.messages import BaseMessage, SystemMessage, AIMessage, HumanMessage

# 维护消息列表
messages: List[BaseMessage] = [
    SystemMessage("你是一位教育工作者，会耐心回复每个有疑问的同学")
]

while True:

    user_input = input("请输入：")

    if user_input == "exit":
        print("会话结束")
        break

    print(f"你：\n\n{user_input}\n")

    # 添加用户消息
    messages.append(HumanMessage(user_input))

    print("老师：\n")

    # 流式输出
    ai_msg = ""
    for chunk in llm.stream(messages):
        print(chunk.text, end="", flush=True)
        ai_msg = ai_msg + chunk.text
    print("\n")

    messages.append(AIMessage(ai_msg))

    # 保留系统提示词，以及最近 5 组对话
    messages = [messages[0]] + messages[1:][-10:]

#### 多模态

In [ ]:
import base64
from pathlib import Path
from langchain_core.messages import HumanMessage

image_path = Path("glass-image.jpg")

image_base64 = base64.b64encode(
    image_path.read_bytes()
).decode("ascii")

resp = llm.invoke([
    HumanMessage(content=[
        {"type": "text", "text": "请描述这张图片的内容"},
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{image_base64}"
            },
        },
    ])
])
print(resp.text)

# 标准化版本
resp = llm.invoke([
    HumanMessage(content_blocks=[
        {"type": "text", "text": "请描述这张图片的内容"},
        {
            "type": "image",
            "base64": image_base64,
            "mime_type": "image/jpeg",
        },
    ])
])
print("\n\n")
print(resp.text)